# Scrublet doublet detection

Run per-PCR-well Scrublet on the Qiu et al. 2024 gastrula-to-pup dataset.

**Input**: `gastrula_to_pup_rna_qc.h5ad` (from NB1: data download and preparation)
**Output**: `scrublet_scores.csv` (cell_id, doublet_score, predicted_doublet)

This is a long-running job (~27h) and should be submitted as a bsub job:
```bash
bsub -q long -n 4 -M 300000 \
  -R"select[mem>300000] rusage[mem=300000] span[hosts=1]" \
  -e ./%J.scrublet.err -o ./%J.scrublet.out -J gastrula_scrublet \
  'PYTHONNOUSERSITE=TRUE module load ISG/conda && conda activate regularizedvi && \
  papermill 01a_scrublet.ipynb 01a_scrublet_out.ipynb'
```

In [ ]:
import gc
import os
import time

import numpy as np
import pandas as pd
import psutil
import scanpy as sc
from tqdm import tqdm

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"

## 1. Load RNA data

In [ ]:
h5ad_path = os.path.join(DATA_DIR, "gastrula_to_pup_rna_qc.h5ad")
print(f"Loading {h5ad_path}...")
adata = sc.read_h5ad(h5ad_path)
print(f"Shape: {adata.shape}, dtype: {adata.X.dtype}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"Unique PCR wells: {adata.obs['pcr_well'].nunique()}")
print(f"RSS: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

## 2. Run Scrublet per PCR well

In sci-RNA-seq3, after step 2 (ligation) all nuclei are pooled and redistributed into
PCR wells. Barcode collisions (doublets) happen when two nuclei sharing the same ligation
barcode land in the same PCR well. `expected_doublet_rate=0.02` is conservative for
~4800 cells/well.

The cache at `scrublet_scores.csv` allows resuming from interrupted runs.

In [ ]:
MIN_CELLS_SCRUBLET = 50
SAVE_EVERY = 200
scrublet_csv = os.path.join(DATA_DIR, "scrublet_scores.csv")

# Load cached results if available
if os.path.exists(scrublet_csv):
    cached = pd.read_csv(scrublet_csv, index_col="cell_id")
    print(f"Loaded cached Scrublet scores: {len(cached):,} cells from {scrublet_csv}")
else:
    cached = pd.DataFrame(columns=["doublet_score", "predicted_doublet"])
    print("No cached Scrublet scores found, computing from scratch")

# Precompute well -> cell indices (O(N) once, vs O(N*K) with per-well masking)
well_groups = adata.obs.groupby("pcr_well").indices
well_counts = {w: len(idx) for w, idx in well_groups.items()}
large_wells = [w for w, c in well_counts.items() if c >= MIN_CELLS_SCRUBLET]
small_wells = [w for w, c in well_counts.items() if c < MIN_CELLS_SCRUBLET]

# Determine which wells still need computation
wells_todo = []
for well in large_wells:
    well_cells = adata.obs_names[well_groups[well]]
    if not well_cells.isin(cached.index).all():
        wells_todo.append(well)

n_small_cells = sum(well_counts[w] for w in small_wells)
n_large_cells = sum(well_counts[w] for w in large_wells)
print(f"Wells >= {MIN_CELLS_SCRUBLET} cells: {len(large_wells)} ({n_large_cells:,} cells)")
print(f"Skipping {len(small_wells)} small wells ({n_small_cells:,} cells)")
print(f"Wells already cached: {len(large_wells) - len(wells_todo)}")
print(f"Wells to compute: {len(wells_todo)}")
print(f"RSS memory before Scrublet: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

# Run Scrublet on remaining wells with timing + memory monitoring
new_results = []
well_times = []
t_start = time.time()

for j, well in enumerate(tqdm(wells_todo, desc="Scrublet per PCR well")):
    t_well = time.time()
    idx = well_groups[well]
    ad_well = adata[idx].copy()
    sc.pp.scrublet(ad_well, expected_doublet_rate=0.02, verbose=False)
    result = ad_well.obs[["doublet_score", "predicted_doublet"]].copy()
    result.index.name = "cell_id"
    new_results.append(result)
    del ad_well

    dt = time.time() - t_well
    well_times.append(dt)

    # Detailed logging for first 20 wells, then every 100
    if j < 20 or (j + 1) % 100 == 0:
        rss_gb = psutil.Process().memory_info().rss / 1e9
        avg_t = np.mean(well_times)
        eta_h = avg_t * (len(wells_todo) - j - 1) / 3600
        print(
            f"  Well {j + 1}/{len(wells_todo)}: {well_counts[well]} cells, "
            f"{dt:.1f}s (avg {avg_t:.1f}s), "
            f"RSS={rss_gb:.1f}GB, ETA={eta_h:.1f}h"
        )

    # Periodic gc
    if (j + 1) % 50 == 0:
        gc.collect()

    # Periodic intermediate save
    if (j + 1) % SAVE_EVERY == 0 and new_results:
        new_df = pd.concat(new_results)
        cached_tmp = pd.concat([cached, new_df])
        cached_tmp = cached_tmp[~cached_tmp.index.duplicated(keep="last")]
        cached_tmp.index.name = "cell_id"
        cached_tmp.to_csv(scrublet_csv)
        print(f"  ** Intermediate save: {len(cached_tmp):,} scores -> {scrublet_csv}")

print(f"\nTotal Scrublet time: {(time.time() - t_start) / 3600:.1f} hours")
if well_times:
    print(f"Average per well: {np.mean(well_times):.1f}s")

# Final merge + save
if new_results:
    new_df = pd.concat(new_results)
    cached = pd.concat([cached, new_df])
    cached = cached[~cached.index.duplicated(keep="last")]
    cached.index.name = "cell_id"
    cached.to_csv(scrublet_csv)
    print(f"Saved {len(cached):,} Scrublet scores to {scrublet_csv}")
    del new_df, new_results
    gc.collect()

## 3. Summary

In [ ]:
# Reload final scores (in case we resumed from cache with nothing new to compute)
scores = pd.read_csv(scrublet_csv, index_col="cell_id")
print(f"Total Scrublet scores: {len(scores):,}")
print(f"Cells in adata: {adata.n_obs:,}")
print(f"Coverage: {100 * scores.index.isin(adata.obs_names).sum() / adata.n_obs:.1f}%")

# Map to adata for summary
adata.obs["doublet_score"] = scores["doublet_score"].reindex(adata.obs_names).values
adata.obs["predicted_doublet"] = scores["predicted_doublet"].reindex(adata.obs_names).fillna(False).values

print("\nDoublet score distribution (non-NaN):")
print(adata.obs["doublet_score"].describe())
print(
    f"\nPredicted doublets: {adata.obs['predicted_doublet'].sum():,} / {adata.n_obs:,} "
    f"({100 * adata.obs['predicted_doublet'].mean():.1f}%)"
)
print(f"Cells with NaN doublet_score (small wells): {adata.obs['doublet_score'].isna().sum():,}")
print(f"\nRSS final: {psutil.Process().memory_info().rss / 1e9:.1f} GB")